# DF Fluorosis — v6 精简实验 (v2.2 复现 + 数据补齐)

**目的**: 用 v2.2 同款代码复现主实验，并导出所有缺失的 per-sample 预测数据用于论文图表。

**3 个实验**:
1. 主实验 5-mode (CE/Cumulative/SORD/EDL/EDL+ORCU), seed=42, 100ep — ~1.5h
2. Lambda Sweep EDL+ORCU (9 combos × 50ep) — ~1.5h
3. Temperature Calibration EDL (T=0.5~5.0) — ~10 min

**关键特性**:
- 每模型导出 per-sample predictions (.npz): logits, probs, evidence, uncertainty, targets
- 自动上传到 Kaggle dataset `hgxiao/fluorosis-data-1-exp-v6`
- 总运行时间: ~3h on T4 x2

## 1. 环境准备

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q
!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project
import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), "Git clone failed!"
print("Code cloned.")

In [ ]:
import os, sys, json, copy, time, itertools
from datetime import datetime
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# 预下载权重
print("Downloading ViT weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Done.")

# 路径
CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
EXPORT_DIR = "/kaggle/working/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("Setup complete.")

## 2. 核心函数 (v2.2 同款)

与 v2.2/kaggle_train_v4.ipynb 完全相同的训练逻辑，额外增加 per-sample export。

In [ ]:
# ============================================================
# Evaluation
# ============================================================
@torch.no_grad()
def evaluate(model, loader, temperature=1.0):
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha, dim=0) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets, temperature=temperature)


# ============================================================
# Per-sample prediction export — 论文图表核心数据
# ============================================================
@torch.no_grad()
def export_predictions(model, loader, filepath):
    """导出逐样本预测，保存为 .npz。
    包含:
      logits (N,4) float16
      probabilities (N,4) float16
      predicted (N,) int8
      targets (N,) int8
      evidence (N,4) float16 — EDL 专有，非 EDL 为全零
      alpha (N,4) float16 — Dirichlet 浓度，仅 EDL
      uncertainty (N,) float16 — Dirichlet u 或 entropy proxy
    """
    model.eval()
    all_z, all_targets = [], []
    all_alpha = []

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
        if alpha is not None:
            all_alpha.append(alpha.cpu())

    z_all = torch.cat(all_z, dim=0)
    targets_all = torch.cat(all_targets, dim=0)
    probs = F.softmax(z_all, dim=-1)
    preds = torch.argmax(probs, dim=-1)
    K = z_all.shape[-1]

    if all_alpha:
        alpha_all = torch.cat(all_alpha, dim=0)
        S = alpha_all.sum(dim=-1)
        u = K / S
        evidence = alpha_all - 1.0
    else:
        alpha_all = None
        evidence = torch.zeros_like(z_all)
        u = -torch.sum(probs * torch.log(probs + 1e-8), dim=-1) / torch.log(torch.tensor(K, dtype=torch.float32))

    data = {
        "logits": z_all.numpy().astype(np.float16),
        "probabilities": probs.numpy().astype(np.float16),
        "predicted": preds.numpy().astype(np.int8),
        "targets": targets_all.numpy().astype(np.int8),
        "evidence": evidence.numpy().astype(np.float16),
        "uncertainty": u.numpy().astype(np.float16),
    }
    if alpha_all is not None:
        data["alpha"] = alpha_all.numpy().astype(np.float16)

    np.savez_compressed(filepath, **data)
    return data


# ============================================================
# Criterion getter (v2.2 同款)
# ============================================================
def get_criterion(mode, model=None, epochs=100):
    if mode == "ce":
        class CELoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                return F.nll_loss(F.log_softmax(z, dim=-1), targets), {"stage": 0, "L_ce": 0}
        return CELoss()
    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cf = CumulativeLoss(num_classes=4)
        class CumLoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                return cf(ol, targets), {"stage": 0, "L_cum": 0}
        return CumLoss()
    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sf = ORCULoss(num_classes=4, t=3.0, lambda_reg=0.0)
        class SordLoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                return sf(z, targets), {"stage": 0, "L_sord": 0}
        return SordLoss()
    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        ef = EDLLoss(num_classes=4, kl_lambda=0.1)
        class EdlLoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                return ef(alpha, targets, epoch, epochs), {"stage": 0, "L_edl": 0}
        return EdlLoss()
    elif mode == "edl_orcu":
        return CombinedLoss(
            num_classes=4, lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0,
            orcu_lambda_reg=0.01,
            stage_1_epochs=5, stage_2_epochs=30, total_epochs=epochs)
    else:
        raise ValueError(f"Unknown mode: {mode}")


# ============================================================
# Optimizer & Scheduler (v2.2 同款)
# ============================================================
def build_opt_sched(model, epochs, lr_bb=1e-4, lr_hd=1e-3, warmup=5, wd=0.05):
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    opt = optim.AdamW([{"params": bb, "lr": lr_bb}, {"params": hd, "lr": lr_hd}], weight_decay=wd)
    ws = LinearLR(opt, start_factor=0.1, total_iters=warmup)
    cs = CosineAnnealingLR(opt, T_max=epochs - warmup)
    sch = SequentialLR(opt, schedulers=[ws, cs], milestones=[warmup])
    return opt, sch


# ============================================================
# 训练函数 (v2.2 同款 + per-sample export + CUDA seed fix)
# ============================================================
def train_one(task, data_root, output_dir, mode="edl", epochs=100,
              batch_size=32, seed=42, export_name=None):
    """训练一个模型。返回 test metrics dict。自动导出 per-sample predictions。"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

    model = build_model(task=task, mode=mode)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params:,} params | {type(model).__name__}")

    criterion = get_criterion(mode, model=model, epochs=epochs)
    opt, sch = build_opt_sched(model, epochs)

    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sch.step()

        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val_acc:
                best_val_acc = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())
            print(f"[E{epoch+1:3d}/{epochs}] Acc={vm['acc']:.4f} F1={vm['macro_f1']:.4f} "
                  f"QWK={vm['qwk']:.4f} ECE={vm['ece']:.4f} Unim={vm['pct_unimodal']:.2%}")

    # Test 评估
    model.load_state_dict(best_state)
    tm = evaluate(model, test_loader)

    # 保存 checkpoint
    torch.save({"model_state_dict": best_state, "test_metrics": tm},
               os.path.join(output_dir, "best.pt"))

    # ★ 导出 per-sample predictions
    ename = export_name or f"{task}_{mode}"
    export_predictions(model, test_loader, os.path.join(EXPORT_DIR, f"{ename}_preds.npz"))
    print(f"  Exported: {ename}_preds.npz")

    flat = {}
    for k, v in tm.items():
        if isinstance(v, (float, int, bool)):
            flat[k] = v
        elif isinstance(v, np.ndarray):
            flat[k] = v.tolist()
    flat["best_val_acc"] = best_val_acc
    flat["n_params"] = n_params
    flat["seed"] = seed
    flat["mode"] = mode
    flat["export_name"] = ename

    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump(flat, f, indent=2)

    print(f"[Test] Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} QWK={tm['qwk']:.4f} "
          f"ECE={tm['ece']:.4f} Unim={tm['pct_unimodal']:.2%} U-ECE={tm.get('u_ece',0):.4f} "
          f"AUROC(u)={tm.get('auroc_u',0):.4f}")
    return flat

print("All core functions ready.")

---
## 实验 1: 主实验 5-Mode 对比

CE / Cumulative / SORD / EDL / EDL+ORCU, seed=42, 100 epochs.
每个模型导出 per-sample predictions 到 `exports/`。

⏱ ~1.5h

In [ ]:
MAIN_MODES = ["ce", "cumulative", "sord", "edl", "edl_orcu"]
main_results = {}
t0 = time.time()

print("=" * 60)
print("实验 1: DF 主实验 5-Mode (seed=42, 100ep)")
print("=" * 60)

for mode in MAIN_MODES:
    print(f"\n{'='*50}")
    print(f"  Mode: {mode}")
    print(f"{'='*50}")
    out_dir = os.path.join(OUTPUT_DIR, f"exp1_{mode}")
    metrics = train_one("df", DATA_ROOT, out_dir, mode=mode, epochs=100,
                        seed=42, export_name=f"exp1_{mode}")
    main_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

with open(os.path.join(OUTPUT_DIR, "exp1_main_results.json"), "w") as f:
    json.dump(main_results, f, indent=2)

print(f"\n实验 1 完成。耗时: {(time.time()-t0)/60:.1f} min")
print(f"\n{'='*80}")
print("DF 主实验结果 (seed=42, 100ep) — v2.2 复现")
print(f"{'='*80}")
print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 78)
for mode, m in main_results.items():
    print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
          f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
          f"{m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
          f"{m.get('pct_unimodal',0):7.2%}")

---
## 实验 2: EDL+ORCU Lambda Sweep

λ_orcu ∈ {0.1, 0.3, 0.5} × λ_reg ∈ {0.005, 0.01, 0.02} = 9 combos, 50 epochs each.

⏱ ~1.5h

In [ ]:
L_ORCU = [0.1, 0.3, 0.5]
L_REG = [0.005, 0.01, 0.02]
SW_EP = 50
SW_SEED = 42

print("=" * 60)
print(f"实验 2: Lambda Sweep — {len(L_ORCU)*len(L_REG)} combos × {SW_EP}ep")
print("=" * 60)

train_loader, val_loader, test_loader = create_dataloaders(
    DATA_ROOT, task="df", batch_size=32, num_workers=2)

sweep_results = []
best_val_acc = 0.0
best_combo = None
t0 = time.time()

for lam_o, lam_r in itertools.product(L_ORCU, L_REG):
    tag = f"lam{lam_o}_reg{lam_r}"
    print(f"\n  [{tag}] training...")
    torch.manual_seed(SW_SEED)
    torch.cuda.manual_seed(SW_SEED)

    model = build_model(task="df", mode="edl_orcu")
    model.to(device)

    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=lam_o, lambda_kl=0.1, orcu_t=3.0,
        orcu_lambda_reg=lam_r,
        stage_1_epochs=5, stage_2_epochs=15, total_epochs=SW_EP)
    opt, sch = build_opt_sched(model, SW_EP)

    best_val = 0.0
    best_state = None
    for epoch in range(SW_EP):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sch.step()
        if (epoch + 1) % 10 == 0 or epoch == SW_EP - 1:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val:
                best_val = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    tm = evaluate(model, test_loader)

    # 导出 per-sample
    export_predictions(model, test_loader,
                       os.path.join(EXPORT_DIR, f"exp2_sweep_{tag}_preds.npz"))

    r = {
        "lambda_orcu": lam_o, "lambda_reg": lam_r,
        "val_acc": best_val, "test_acc": tm["acc"], "test_qwk": tm["qwk"],
        "test_ece": tm["ece"], "test_unim": tm["pct_unimodal"],
        "test_u_ece": tm["u_ece"], "test_auroc_u": tm["auroc_u"],
        "test_f1": tm["macro_f1"],
    }
    sweep_results.append(r)
    if best_val > best_val_acc:
        best_val_acc = best_val
        best_combo = (lam_o, lam_r)
    print(f"    Val={best_val:.4f} Test={tm['acc']:.4f} QWK={tm['qwk']:.4f} ECE={tm['ece']:.4f}")

with open(os.path.join(OUTPUT_DIR, "exp2_lambda_sweep.json"), "w") as f:
    json.dump({"results": sweep_results, "best_combo": list(best_combo)}, f, indent=2)

print(f"\n实验 2 完成。耗时: {(time.time()-t0)/60:.1f} min")
print(f"\nLambda Sweep 结果 (按 Val Acc 排序):")
print(f"{'λ_orcu':>8} {'λ_reg':>8} {'Val':>8} {'Test':>8} {'QWK':>8} {'F1':>8} {'ECE':>8} {'Unim':>8}")
print("-" * 72)
for r in sorted(sweep_results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
          f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} {r['test_f1']:8.4f} "
          f"{r['test_ece']:8.4f} {r['test_unim']:7.2%}")
print(f"\nBest: λ_orcu={best_combo[0]}, λ_reg={best_combo[1]} (Val={best_val_acc:.4f})")

---
## 实验 3: EDL Temperature Calibration

用实验 1 的 EDL 模型 (exp1_edl/best.pt), 扫 T ∈ {0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0}。

⏱ ~10 min

In [ ]:
print("=" * 60)
print("实验 3: Temperature Calibration — DF EDL")
print("=" * 60)

ckpt_path = os.path.join(OUTPUT_DIR, "exp1_edl", "best.pt")
if not os.path.exists(ckpt_path):
    print(f"ERROR: {ckpt_path} not found! Run Experiment 1 first.")
else:
    model = build_model(task="df", mode="edl")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()
    print(f"Loaded EDL from {ckpt_path}")

    _, _, test_loader = create_dataloaders(DATA_ROOT, task="df", batch_size=32, num_workers=2)

    TEMPS = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
    temp_results = {}

    print(f"{'T':>6} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
    print("-" * 72)
    for T in TEMPS:
        m = evaluate(model, test_loader, temperature=T)
        temp_results[f"T_{T}"] = {
            "acc": m["acc"], "macro_f1": m["macro_f1"], "qwk": m["qwk"],
            "ece": m["ece"], "u_ece": m["u_ece"], "auroc_u": m["auroc_u"],
            "pct_unimodal": m["pct_unimodal"],
        }
        print(f"{T:6.1f} {m['acc']:8.4f} {m['macro_f1']:8.4f} {m['qwk']:8.4f} "
              f"{m['ece']:8.4f} {m['u_ece']:8.4f} {m['auroc_u']:8.4f} {m['pct_unimodal']:7.2%}")

    best_ece = min(temp_results.items(), key=lambda x: x[1]["ece"])
    best_auroc = max(temp_results.items(), key=lambda x: x[1]["auroc_u"])
    print(f"\nBest ECE: {best_ece[0]} (ECE={best_ece[1]['ece']:.4f})")
    print(f"Best AUROC(u): {best_auroc[0]} (AUROC(u)={best_auroc[1]['auroc_u']:.4f})")

    with open(os.path.join(OUTPUT_DIR, "exp3_temperature.json"), "w") as f:
        json.dump(temp_results, f, indent=2)

print("\n实验 3 完成。")

---
## 结果汇总

打印所有实验结果，列出导出的 per-sample 文件。

In [ ]:
print("=" * 80)
print("V6 实验结果汇总")
print(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# 实验 1
p1 = os.path.join(OUTPUT_DIR, "exp1_main_results.json")
if os.path.exists(p1):
    with open(p1) as f:
        exp1 = json.load(f)
    print("\n--- 实验 1: 主实验 5-Mode ---")
    print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
    print("-" * 78)
    for mode, m in exp1.items():
        print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
              f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
              f"{m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
              f"{m.get('pct_unimodal',0):7.2%}")

# 实验 2
p2 = os.path.join(OUTPUT_DIR, "exp2_lambda_sweep.json")
if os.path.exists(p2):
    with open(p2) as f:
        exp2 = json.load(f)
    print(f"\n--- 实验 2: Lambda Sweep (best: λ_orcu={exp2['best_combo'][0]}, λ_reg={exp2['best_combo'][1]}) ---")
    for r in sorted(exp2["results"], key=lambda x: x["val_acc"], reverse=True):
        print(f"  λ={r['lambda_orcu']:.2f} reg={r['lambda_reg']:.4f}: "
              f"Test Acc={r['test_acc']:.4f} QWK={r['test_qwk']:.4f} ECE={r['test_ece']:.4f}")

# 实验 3
p3 = os.path.join(OUTPUT_DIR, "exp3_temperature.json")
if os.path.exists(p3):
    with open(p3) as f:
        exp3 = json.load(f)
    print("\n--- 实验 3: Temperature Calibration ---")
    for Tk in sorted(exp3.keys(), key=lambda k: float(k.replace("T_",""))):
        r = exp3[Tk]
        print(f"  {Tk}: Acc={r['acc']:.4f} ECE={r['ece']:.4f} AUROC(u)={r['auroc_u']:.4f}")

# Per-sample 文件清单
export_files = sorted([f for f in os.listdir(EXPORT_DIR) if f.endswith(".npz")])
print(f"\n--- Per-Sample Predictions: {len(export_files)} 个文件 ---")
for f in export_files:
    fsize = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1024
    print(f"  {f} ({fsize:.1f} KB)")

print("\n=== 全部实验完成 ===")

---
## 导出到 Kaggle Dataset `fluorosis-data-1-exp-v6`

打包所有结果（JSON + NPZ + checkpoint），创建/更新 Kaggle dataset。
下载链接会打印在输出中。

In [ ]:
import shutil

PKG = "/kaggle/working/fluorosis-data-1-exp-v6"
os.makedirs(PKG, exist_ok=True)

collected = []

# 结果 JSONs
for fname in ["exp1_main_results.json", "exp2_lambda_sweep.json", "exp3_temperature.json"]:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(PKG, fname))
        collected.append(fname)

# 模型 checkpoints (仅核心的)
for mode in ["ce", "cumulative", "sord", "edl", "edl_orcu"]:
    src = os.path.join(OUTPUT_DIR, f"exp1_{mode}", "best.pt")
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(PKG, f"checkpoint_{mode}.pt"))
        collected.append(f"checkpoint_{mode}.pt")

# Per-sample predictions (所有 .npz)
for fname in os.listdir(EXPORT_DIR):
    if fname.endswith(".npz"):
        shutil.copy2(os.path.join(EXPORT_DIR, fname), os.path.join(PKG, fname))
        collected.append(fname)

# Dataset metadata
metadata = {
    "title": "fluorosis-data-1-exp-v6",
    "id": "hgxiao/fluorosis-data-1-exp-v6",
    "licenses": [{"name": "CC0-1.0"}],
    "description": (
        "DF fluorosis experiment results v6 (v2.2 reproduction + missing data). "
        "Contains: 5-mode main comparison, EDL+ORCU lambda sweep (9 combos), "
        "EDL temperature calibration (7 temps), "
        "per-sample predictions for ALL models (logits, probs, evidence, uncertainty). "
        "ViT-Base (ImageNet-21k), DFID dataset. "
        f"Generated {datetime.now().strftime('%Y-%m-%d')}."
    ),
}
with open(os.path.join(PKG, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# 复制 notebook
nb_src = "/kaggle/working/fluorosis_project/Kaggle_Setup/kaggle_train_v6.ipynb"
if os.path.exists(nb_src):
    shutil.copy2(nb_src, os.path.join(PKG, "kaggle_train_v6.ipynb"))
    collected.append("kaggle_train_v6.ipynb")

print(f"共打包 {len(collected)} 个文件:")
total_mb = 0
for f in sorted(collected):
    fsize = os.path.getsize(os.path.join(PKG, f)) / 1024
    total_mb += fsize / 1024
    if fsize > 1024:
        print(f"  {f} ({fsize/1024:.1f} MB)")
    else:
        print(f"  {f} ({fsize:.1f} KB)")
print(f"\n总大小: {total_mb:.1f} MB")
print(f"\n{'='*60}")
print("上传到 Kaggle Dataset:")
print(f"{'='*60}")
print(f"\n  kaggle datasets create -p {PKG} --dir-mode zip")
print(f"\n或更新已有 dataset:")
print(f"  kaggle datasets version -p {PKG} -m 'v6: Complete DF experiment data' --dir-mode zip")
print(f"\nPackage ready at: {PKG}")

In [ ]:
# 自动上传 (如果 kaggle CLI 可用)
import subprocess

PKG = "/kaggle/working/fluorosis-data-1-exp-v6"
try:
    result = subprocess.run(["kaggle", "datasets", "list", "--user", "hgxiao"],
                           capture_output=True, text=True, timeout=30)
    if "fluorosis-data-1-exp-v6" in result.stdout:
        print("Dataset 已存在，创建新版本...")
        subprocess.run(["kaggle", "datasets", "version", "-p", PKG,
                       "-m", f"v6: Complete DF experiment data ({datetime.now().strftime('%Y-%m-%d')})"],
                      check=True, timeout=120)
    else:
        print("创建新 dataset...")
        subprocess.run(["kaggle", "datasets", "create", "-p", PKG, "--dir-mode", "zip"],
                      check=True, timeout=120)
    print("上传成功!")
    print("https://www.kaggle.com/datasets/hgxiao/fluorosis-data-1-exp-v6")
except subprocess.CalledProcessError as e:
    print(f"上传失败: {e}")
    print("请手动执行上一个 cell 中打印的命令。")
except FileNotFoundError:
    print("Kaggle CLI 不可用。请手动执行上一个 cell 中打印的命令。")
except Exception as e:
    print(f"错误: {e}")

---
## 数据清单 (download 后交给 Claude)

下载 Kaggle dataset `hgxiao/fluorosis-data-1-exp-v6` 后，你会得到以下文件：

**结果 JSON (3 个)**:
- `exp1_main_results.json` — 5-mode 聚合指标
- `exp2_lambda_sweep.json` — 9-combo lambda sweep
- `exp3_temperature.json` — 7-temp calibration

**模型 Checkpoint (5 个)**:
- `checkpoint_ce.pt`, `checkpoint_cumulative.pt`, `checkpoint_sord.pt`, `checkpoint_edl.pt`, `checkpoint_edl_orcu.pt`

**Per-Sample Predictions (14 个 .npz)**:
- `exp1_ce_preds.npz` — CE 逐样本预测
- `exp1_cumulative_preds.npz` — Cumulative
- `exp1_sord_preds.npz` — SORD
- `exp1_edl_preds.npz` — EDL (有 alpha/evidence)
- `exp1_edl_orcu_preds.npz` — EDL+ORCU
- `exp2_sweep_lam*_reg*_preds.npz` — 9 个 sweep 预测

每个 .npz 包含: `logits`, `probabilities`, `predicted`, `targets`, `evidence`, `uncertainty` (EDL 额外有 `alpha`)。

下载到项目后告诉我，我会用这些数据生成论文全部图表。